In [1]:
!pip install mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.1/36.1 MB 51.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 19.7 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:
      Successfully uninstalled protobuf-3.20.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
apache-beam 2.46.0 requires cloudpickle~=2.2.1, but you have cloudpickle 3.0.0 which is incompatible.
apache-beam 2.46.0 requires dill<0.3.2,>=0.3.1.1, but you have dill 0.3.8 which is incompatible.
apache-beam 2.46.0 requires numpy<1.25.0,>=1.14.3, but you have numpy 1.26.4 which is incompatible.
apache-beam 2.46.0 requires protobuf<4,>3.12.2, but you have protobuf 4.25.5 which is incompatible.
apache-beam 2.46.0 requires pyarrow<10.0.0,>=3.0.0, but you have pyarrow 16.1.0 which is inc

### Extract 2D Skeleton Pose

In [ ]:
import cv2
import mediapipe as mp

# Khởi tạo mediapipe pose
mp_pose = mp.solutions.pose
pose = mp_pose.Pose()

# Khởi tạo VideoCapture để đọc video
cap = cv2.VideoCapture('/kaggle/input/large-gait-dataset/Gait Dataset Video/P10_E2T4_90.mp4')

# Lấy thông tin về chiều cao, chiều rộng và FPS của video gốc
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# Khởi tạo video writer để xuất ra video 2D với kích thước tương tự video gốc
output_video = cv2.VideoWriter('output_2d_video.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))

# Đọc từng frame của video và trích xuất pose landmarks
while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    # Chuyển ảnh sang RGB (Mediapipe cần định dạng RGB để xử lý)
    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Phát hiện pose landmarks
    results = pose.process(image_rgb)
    
    # Nếu phát hiện pose landmarks
    if results.pose_landmarks:
        landmarks = results.pose_landmarks.landmark
        
        # Chuyển đổi tọa độ từ tỷ lệ sang tọa độ pixel trên khung hình
        h, w, _ = frame.shape
        for lm in landmarks:
            cx, cy = int(lm.x * w), int(lm.y * h)
            # Vẽ điểm landmarks (màu xanh lá cây và kích thước 5px)
            cv2.circle(frame, (cx, cy), 3, (0, 255, 0), -1)
        
        # Vẽ đường nối các khớp của cơ thể (pose connections)
        connections = [
            (11, 13), (13, 15),  # Cánh tay trái
            (12, 14), (14, 16),  # Cánh tay phải
            (23, 25), (25, 27),  # Chân trái
            (24, 26), (26, 28),  # Chân phải
            (11, 12), (23, 24), (11, 23), (12, 24)  # Thân người
        ]
        
        for connection in connections:
            x1, y1 = int(landmarks[connection[0]].x * w), int(landmarks[connection[0]].y * h)
            x2, y2 = int(landmarks[connection[1]].x * w), int(landmarks[connection[1]].y * h)
            # Vẽ đường nối (màu đỏ và độ dày 2px)
            cv2.line(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)

    # Viết frame đã vẽ landmarks vào video
    output_video.write(frame)

# Đóng các file video sau khi xử lý xong
cap.release()
output_video.release()

print("Xuất video 2D hoàn thành!")


### Extract 3D Skeleton Pose

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt

# Đặt trực tiếp đường dẫn video và tên file đầu ra
input_video = '/kaggle/input/demo-gait-video/P3_E1T2_45.mp4'  # Đường dẫn đến video
output_video_path = 'output_3d_video.mp4'  # Tên file video đầu ra
verbose = False  # Nếu muốn in ra thông tin chi tiết, đặt thành True

# Khởi tạo mediapipe pose
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

# Đọc video đầu vào
cap = cv2.VideoCapture(input_video)

# Lấy thông tin chiều rộng, chiều cao, và FPS của video
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# Khởi tạo VideoWriter để ghi video đầu ra
output_video = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (640, 480))

frame = 0
max_frames = 5000

# Hàm vẽ landmarks 3D và nối các điểm bằng POSE_CONNECTIONS
def plot_3d_landmarks(landmarks, connections):
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    ax.set_xlim([-1, 1])
    ax.set_ylim([-1, 1])
    ax.set_zlim([-1, 1])

    # Tạo các danh sách x, y, z cho từng landmark
    xs = [lm.x for lm in landmarks]
    ys = [lm.y for lm in landmarks]
    zs = [lm.z for lm in landmarks]
    
    # Vẽ các điểm landmarks
    ax.scatter(xs, ys, zs, color='blue')

    # Nối các điểm bằng POSE_CONNECTIONS
    for connection in connections:
        start_idx, end_idx = connection
        x_line = [xs[start_idx], xs[end_idx]]
        y_line = [ys[start_idx], ys[end_idx]]
        z_line = [zs[start_idx], zs[end_idx]]
        ax.plot(x_line, y_line, z_line, color='red')

    plt.close(fig)
    
    # Lưu hình ảnh matplotlib vào buffer
    fig.canvas.draw()
    img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    
    return cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

# Khởi tạo Pose model của Mediapipe
with mp_pose.Pose(
        min_detection_confidence=0.75,
        min_tracking_confidence=0.6) as pose:

    while cap.isOpened():
        success, image = cap.read()

        if not success:
            break

        # Chuyển ảnh sang RGB để Mediapipe xử lý
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Đánh dấu ảnh là không ghi lại để tăng tốc độ xử lý
        image.flags.writeable = False
        results = pose.process(image)

        if results.pose_world_landmarks:
            # Vẽ landmarks 3D từ pose_world_landmarks với POSE_CONNECTIONS
            image_3d = plot_3d_landmarks(results.pose_world_landmarks.landmark, mp_pose.POSE_CONNECTIONS)

            # Resize hình ảnh cho khớp với kích thước video đầu ra
            image_3d_resized = cv2.resize(image_3d, (640, 480))

            # Ghi frame đã xử lý vào video đầu ra
            output_video.write(image_3d_resized)

        frame += 1
        if frame > max_frames:
            break

# Giải phóng video capture và video writer
cap.release()
output_video.release()

print(f"Xuất video 3D hoàn thành: {output_video_path}")


### Extract Feature x, y, z, v on Mediapipe

In [ ]:
import cv2
import mediapipe as mp
import csv

# Khởi tạo mediapipe pose
mp_pose = mp.solutions.pose
pose = mp_pose.Pose()

# Khởi tạo VideoCapture để đọc video
cap = cv2.VideoCapture('/kaggle/input/demo-gait-video/P3_E1T2_45.mp4')

# Tạo file CSV để lưu tọa độ pose landmarks
with open('pose_landmarks.csv', mode='w', newline='') as file:
    writer = csv.writer(file)
    
    # Ghi tiêu đề (header) với tên từng landmark
    header = ['frame'] + [f'{name}_{axis}' for name in range(33) for axis in ['x', 'y', 'z', 'visibility']]
    writer.writerow(header)

    frame_index = 0
    
    # Đọc từng frame của video và trích xuất pose landmarks
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        # Chuyển ảnh sang RGB (Mediapipe cần định dạng RGB để xử lý)
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Phát hiện pose landmarks
        results = pose.process(image_rgb)

        # Nếu phát hiện pose landmarks
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            
            # Tạo danh sách chứa tọa độ của tất cả các landmark trong frame này
            frame_data = [frame_index]
            for lm in landmarks:
                frame_data.extend([lm.x, lm.y, lm.z, lm.visibility])
            
            # Ghi dữ liệu landmarks của frame hiện tại vào file CSV
            writer.writerow(frame_data)
        
        frame_index += 1

# Đóng video sau khi xử lý xong
cap.release()

print("Xuất dữ liệu pose landmarks ra file CSV hoàn thành!")


### Extract Feature x, y on Mediapipe

In [5]:
import cv2
import mediapipe as mp
import csv
import time

def extract_pose_landmarks_xy(file_path, save_path, save_img_path):
    # Khởi tạo mediapipe pose
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose()

    # Khởi tạo VideoCapture để đọc video
    cap = cv2.VideoCapture(file_path)

    # Lấy FPS của video từ thông tin video đầu vào
    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f"FPS của video: {fps}")
    
    # Tạo thư mục nếu chưa tồn tại
    if not os.path.exists(save_img_path):
        os.makedirs(save_img_path)

    # Mở file CSV để lưu tọa độ pose landmarks
    with open(save_path, mode='w', newline='') as file:
        writer = csv.writer(file)
        
        # Ghi tiêu đề (header) với tên từng landmark
        header = ['frame'] + [f'{name}_{axis}' for name in range(33) for axis in ['x', 'y']]
        writer.writerow(header)

        frame_index = 0
        all_frame = 0
        start_time = time.time()  # Bắt đầu đo thời gian

        # Đọc từng frame của video và trích xuất pose landmarks
        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                break

            # Chuyển ảnh sang RGB (Mediapipe cần định dạng RGB để xử lý)
            image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # Phát hiện pose landmarks
            results = pose.process(image_rgb)

            # Nếu phát hiện pose landmarks
            if results.pose_landmarks:
                landmarks = results.pose_landmarks.landmark
                
                # Tạo danh sách chứa tọa độ x, y của tất cả các landmark trong frame này
                frame_data = [all_frame]
                for lm in landmarks:
                    frame_data.extend([lm.x, lm.y])
                
                # Ghi dữ liệu landmarks của frame hiện tại vào file CSV
                writer.writerow(frame_data)
                
                gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                frame_file_path = f"{save_img_path}/frame_{all_frame:04d}.png"
                cv2.imwrite(frame_file_path, gray_frame)
            
            frame_index += 1
            all_frame += 1
            # Đo thời gian đã trôi qua
            elapsed_time = time.time() - start_time
            if elapsed_time >= 1.0:
#                 print(f"Số frame xử lý trong 1 giây: {frame_index}")
                frame_index = 0
                start_time = time.time()  # Reset thời gian đo

    # Đóng video sau khi xử lý xong
    cap.release()
    print(f"Extract pose landmarks {file_path} -> {save_path} ✅!")

# Ví dụ sử dụng hàm:
# file_path = '/kaggle/input/demo-gait-video/P3_E1T2_45.mp4'
# save_path = 'pose_landmarks_xy.csv'
# extract_pose_landmarks_xy(file_path, save_path)


### Extract Feature Distance Matrix

In [6]:
import cv2
import mediapipe as mp
import csv
import os
import time
import math

def extract_pose_landmarks_distance_matrix(file_path, save_path, save_img_path):
    # Khởi tạo mediapipe pose
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose()

    # Khởi tạo VideoCapture để đọc video
    cap = cv2.VideoCapture(file_path)

    # Lấy FPS của video từ thông tin video đầu vào
    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f"FPS của video: {fps}")
    
    # Tạo thư mục nếu chưa tồn tại
    if not os.path.exists(save_img_path):
        os.makedirs(save_img_path)

    # Mở file CSV để lưu ma trận khoảng cách
    with open(save_path, mode='w', newline='') as file:
        writer = csv.writer(file)
        
        # Ghi tiêu đề (header) với tên từng cặp landmark
        header = ['frame'] + [f'dist_{i}-{j}' for i in range(33) for j in range(i+1, 33)]
        writer.writerow(header)

        frame_index = 0
        all_frame = 0
        start_time = time.time()  # Bắt đầu đo thời gian

        # Đọc từng frame của video và trích xuất pose landmarks
        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                break

            # Chuyển ảnh sang RGB (Mediapipe cần định dạng RGB để xử lý)
            image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # Phát hiện pose landmarks
            results = pose.process(image_rgb)

            # Nếu phát hiện pose landmarks
            if results.pose_landmarks:
                landmarks = results.pose_landmarks.landmark
                
                # Tạo danh sách chứa khoảng cách giữa tất cả các cặp landmarks trong frame này
                frame_data = [all_frame]
                for i in range(len(landmarks)):
                    for j in range(i+1, len(landmarks)):
                        # Tính khoảng cách Euclid giữa landmark i và j
                        dist = math.sqrt((landmarks[i].x - landmarks[j].x)**2 + (landmarks[i].y - landmarks[j].y)**2)
                        frame_data.append(dist)
                
                # Ghi dữ liệu khoảng cách của frame hiện tại vào file CSV
                writer.writerow(frame_data)
                
                gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                frame_file_path = f"{save_img_path}/frame_{all_frame:04d}.png"
                cv2.imwrite(frame_file_path, gray_frame)
            
            frame_index += 1
            all_frame += 1
            # Đo thời gian đã trôi qua
            elapsed_time = time.time() - start_time
            if elapsed_time >= 1.0:
                frame_index = 0
                start_time = time.time()  # Reset thời gian đo

    # Đóng video sau khi xử lý xong
    cap.release()
    print(f"Extract pose landmarks distance matrix {file_path} -> {save_path} ✅!")

# Ví dụ sử dụng hàm:
# file_path = '/kaggle/input/demo-gait-video/P3_E1T2_45.mp4'
# save_path = 'pose_landmarks_distance_matrix.csv'
# extract_pose_landmarks_distance_matrix(file_path, save_path)


### Extract Feature Distance Matrix not Save Frame

In [7]:
import cv2
import mediapipe as mp
import csv
import os
import time
import math

def extract_pose_landmarks_distance_matrix_no_images(file_path, save_path, save_img_path):
    # Khởi tạo mediapipe pose
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose()

    # Khởi tạo VideoCapture để đọc video
    cap = cv2.VideoCapture(file_path)

    # Lấy FPS của video từ thông tin video đầu vào
    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f"FPS của video: {fps}")
    
    # Tạo thư mục nếu chưa tồn tại
#     if not os.path.exists(save_img_path):
#         os.makedirs(save_img_path)

    # Mở file CSV để lưu ma trận khoảng cách
    with open(save_path, mode='w', newline='') as file:
        writer = csv.writer(file)
        
        # Ghi tiêu đề (header) với tên từng cặp landmark
        header = ['frame'] + [f'dist_{i}-{j}' for i in range(33) for j in range(i+1, 33)]
        writer.writerow(header)

        frame_index = 0
        all_frame = 0
        start_time = time.time()  # Bắt đầu đo thời gian

        # Đọc từng frame của video và trích xuất pose landmarks
        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                break

            # Chuyển ảnh sang RGB (Mediapipe cần định dạng RGB để xử lý)
            image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # Phát hiện pose landmarks
            results = pose.process(image_rgb)

            # Nếu phát hiện pose landmarks
            if results.pose_landmarks:
                landmarks = results.pose_landmarks.landmark
                
                # Tạo danh sách chứa khoảng cách giữa tất cả các cặp landmarks trong frame này
                frame_data = [all_frame]
                for i in range(len(landmarks)):
                    for j in range(i+1, len(landmarks)):
                        # Tính khoảng cách Euclid giữa landmark i và j
                        dist = math.sqrt((landmarks[i].x - landmarks[j].x)**2 + (landmarks[i].y - landmarks[j].y)**2)
                        frame_data.append(dist)
                
                # Ghi dữ liệu khoảng cách của frame hiện tại vào file CSV
                writer.writerow(frame_data)
                
                gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
#                 frame_file_path = f"{save_img_path}/frame_{all_frame:04d}.png"
#                 cv2.imwrite(frame_file_path, gray_frame)
            
            frame_index += 1
            all_frame += 1
            # Đo thời gian đã trôi qua
            elapsed_time = time.time() - start_time
            if elapsed_time >= 1.0:
                frame_index = 0
                start_time = time.time()  # Reset thời gian đo

    # Đóng video sau khi xử lý xong
    cap.release()
    print(f"Extract pose landmarks distance matrix {file_path} -> {save_path} ✅!")

# Ví dụ sử dụng hàm:
# file_path = '/kaggle/input/demo-gait-video/P3_E1T2_45.mp4'
# save_path = 'pose_landmarks_distance_matrix.csv'
# extract_pose_landmarks_distance_matrix(file_path, save_path)


### Extract Person

In [ ]:
import os

# Định dạng video cần tìm
video_extensions = ('.mp4', '.avi', '.mov', '.mkv')
all_video_path = []

folder_path = "/kaggle/input/large-gait-dataset/Gait Dataset Video"
    
# Lấy danh sách tất cả các video trong folder
video_paths = [os.path.join(folder_path, file) for file in os.listdir(folder_path) if file.endswith(video_extensions)]
all_video_path.extend(video_paths)

In [10]:
len(all_video_path)

2944

In [11]:
from tqdm import tqdm

In [12]:
import os
import shutil

output_dirs_rm = ['/kaggle/working/Gait Dataset Video']

for output_dir_rm in output_dirs_rm:
    # Kiểm tra nếu thư mục tồn tại
    if os.path.exists(output_dir_rm):
        # Xóa tất cả các file trong thư mục
        for file in os.listdir(output_dir_rm):
            file_path = os.path.join(output_dir_rm, file)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.unlink(file_path)  # Xóa file hoặc symbolic link
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)  # Xóa thư mục con
            except Exception as e:
                print(f"Không thể xóa {file_path} do lỗi: {e}")
    else:
        print("Thư mục output không tồn tại.")


Thư mục output không tồn tại.


In [13]:
import os

file_path = '/kaggle/working/output_folders1.zip'

# Kiểm tra nếu file tồn tại
if os.path.exists(file_path):
    os.remove(file_path)  # Xóa file
    print(f"Đã xóa file: {file_path}")
else:
    print("File không tồn tại.")

File không tồn tại.


In [14]:
person_solve = [f"P{i}" for i in range(46, 51)]
person_solve

['P46', 'P47', 'P48', 'P49', 'P50']

In [15]:
# person_solve = ["P1", "P2", "P3", "P4", "P5", "P6", "P7", "P8", "P9", "P10", "P11", "P12", "P13", "P14", "P15", "P16", "P17", "P18"]
for vid in tqdm(all_video_path):
    output = vid.split("/")[-1][:-4]
    folder = vid.split("/")[-2]
    id_person = output[:-4].split("_")[0]
    output_dir = f"/kaggle/working/{folder}/{id_person}"
    output_path = os.path.join(output_dir, output + ".csv")
    output_img_path = os.path.join(output_dir, output + "_img")
    
    check = False
    for person in person_solve:
        if id_person == person:
            check = True
    if check == True:
        print(f"Xử lý {output_path}")
#     Tạo thư mục nếu chưa tồn tại
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
#         extract_pose_landmarks_xy(vid, output_path, output_img_path)
        extract_pose_landmarks_distance_matrix_no_images_with_GPU(vid, output_path, output_img_path)

  0%|          | 0/2944 [00:00<?, ?it/s]

Xử lý /kaggle/working/Gait Dataset Video/P46/P46_E3T1_Front.csv
FPS của video: 30.0


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1730723989.276260      94 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1730723989.342520      94 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1730723989.375039      94 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
  0%|          | 13/2944 [00:07<29:15,  1.67it/s]

Extract pose landmarks distance matrix /kaggle/input/large-gait-dataset/Gait Dataset Video/P46_E3T1_Front.mp4 -> /kaggle/working/Gait Dataset Video/P46/P46_E3T1_Front.csv ✅!
Xử lý /kaggle/working/Gait Dataset Video/P50/P50_E1T1_45.csv
FPS của video: 30.0


W0000 00:00:1730723997.044436     105 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1730723997.096062     105 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
  1%|          | 15/2944 [00:15<58:29,  1.20s/it]

...

Extract pose landmarks distance matrix /kaggle/input/large-gait-dataset/Gait Dataset Video/P50_E2T3_90.mp4 -> /kaggle/working/Gait Dataset Video/P50/P50_E2T3_90.csv ✅!
Xử lý /kaggle/working/Gait Dataset Video/P46/P46_E1T3_90.csv
FPS của video: 30.0


W0000 00:00:1730725648.109114    2003 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1730725648.163773    2003 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
100%|█████████▉| 2939/2944 [27:46<00:01,  2.90it/s]

Extract pose landmarks distance matrix /kaggle/input/large-gait-dataset/Gait Dataset Video/P46_E1T3_90.mp4 -> /kaggle/working/Gait Dataset Video/P46/P46_E1T3_90.csv ✅!
Xử lý /kaggle/working/Gait Dataset Video/P50/P50_E1T5_90.csv
FPS của video: 30.0


W0000 00:00:1730725655.672088    2008 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1730725655.720146    2008 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
100%|██████████| 2944/2944 [27:54<00:00,  1.76it/s]

Extract pose landmarks distance matrix /kaggle/input/large-gait-dataset/Gait Dataset Video/P50_E1T5_90.mp4 -> /kaggle/working/Gait Dataset Video/P50/P50_E1T5_90.csv ✅!


### Zip feature

In [16]:
import os
import zipfile

# Đường dẫn đến thư mục gốc
working_dir = '/kaggle/working/Gait Dataset Video'

# Đường dẫn file zip muốn tạo ra
output_zip_file = '/kaggle/working/output_folders1.zip'

# Tạo tệp zip
with zipfile.ZipFile(output_zip_file, 'w') as zipf:
    for folder in person_solve:
        folder_path = os.path.join(working_dir, folder)
        print(folder_path)
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                # Thêm file vào zip (với cấu trúc thư mục)
                zipf.write(file_path, os.path.relpath(file_path, working_dir))

print(f"Đã nén các folder {person_solve} thành công!")


/kaggle/working/Gait Dataset Video/P46
/kaggle/working/Gait Dataset Video/P47
/kaggle/working/Gait Dataset Video/P48
/kaggle/working/Gait Dataset Video/P49
/kaggle/working/Gait Dataset Video/P50
Đã nén các folder ['P46', 'P47', 'P48', 'P49', 'P50'] thành công!


In [17]:
print()